# Localização da aorta com Frangi de sigmas altos

O experimento testa se realçar estruturas de maior escala antes da Hough ajuda a
localizar a aorta. Os dois caminhos usam a mesma tomografia pré-processada, o
mesmo filtro de trajetória e o mesmo level set. A entrada da Hough muda entre
a tomografia após a maior componente conectada (LCC) e o mapa Frangi adicional.

A comparação usa um exame de treino por vez. No MM-WHS, o Dice é medido contra
o rótulo 820 da aorta ascendente. No ImageCAS, contornos, círculos e volume
servem para inspeção qualitativa.

In [ ]:
# ruff: noqa: E402
import json
import sys
from functools import partial
from pathlib import Path
from time import perf_counter

import pandas as pd
from IPython.display import display

notebook_dir = Path.cwd().resolve()
for candidate in (notebook_dir, *notebook_dir.parents):
    src_dir = candidate / "src"
    if (src_dir / "utils").is_dir():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break

from external_ccta_batch_pipeline import resolve_dataset_path
from utils.project.analysis.aorta_vesselness import (
    evaluate_aorta_localization,
    normalize_aorta_sigmas,
)
from utils.project.datasets.ccta import (
    discover_ccta_dataset,
    load_ccta_aorta_ground_truth,
    load_ccta_volume,
)
from utils.project.runtime.notebook_env import (
    configure_notebook_environment,
    load_notebook_pipeline_config,
    resolve_imagecas_base_path,
)
from utils.segmentation.pipeline.preprocessing import (
    load_and_preprocess_image,
    preprocess_ccta_volume,
)
from utils.visualization.images.image_slices import plot_mip_projection
from utils.visualization.images.volume import visualize_3d_k3d

REPO_ROOT = configure_notebook_environment(chdir_to_src=False)

## 1. Configuração

Use `EXAM_ID = None` para selecionar o caso padrão do banco ou informe um ID de
treino. Configure `MMWHS_BASE_PATH` ou `IMAGECAS_BASE_PATH` se os dados estiverem
fora dos caminhos locais conhecidos. A opção `USE_GPU` solicita GPU apenas para
o cálculo do Frangi. Hough e level set rodam na CPU.

Os sigmas estão em voxels da grade processada, inclusive em `high`. O
espaçamento abaixo indica quanto cada voxel representa em milímetros por eixo.
As escalas são valores exploratórios e podem ser editadas.

O experimento usa o sigma `16` por padrão. A seção 4 preserva as comparações
que levaram à escolha. A tabela principal salva antes da troca usou os quatro
sigmas de referência.

In [ ]:
DATASET = "imagecas"  # "mmwhs" ou "imagecas"
EXAM_ID = 752  # Exemplos: "ct_train_1001" ou 90
RESOLUTION = "mid"  # "mid" ou "high"
FRANGI_SIGMAS = (16,)
USE_GPU = False

default_exam_ids = {"mmwhs": "ct_train_1001", "imagecas": "90"}
if DATASET not in default_exam_ids:
    raise ValueError("DATASET deve ser 'mmwhs' ou 'imagecas'.")

exam_id = default_exam_ids[DATASET] if EXAM_ID is None else str(EXAM_ID)
sigmas = normalize_aorta_sigmas(FRANGI_SIGMAS)

config = load_notebook_pipeline_config(
    REPO_ROOT / "config/pipeline_config.json",
    RESOLUTION,
)
frangi_config = {
    **config["VESSELNESS_AORTA"],
    "gamma": None,
    "normalization": "minmax",
}

## 2. Pré-processamento compartilhado

O volume passa pela redução de resolução, limiarização e seleção da LCC com a
configuração do pipeline. No MM-WHS, a máscara binária da aorta ascendente
acompanha o volume e é reamostrada por vizinho mais próximo.

In [ ]:
if DATASET == "mmwhs":
    base_path = resolve_dataset_path("mmwhs", None)
    inventory = discover_ccta_dataset(DATASET, base_path)
    matches = inventory.loc[
        inventory["subset"].eq("train") & inventory["exam_id"].eq(exam_id)
    ]
    if matches.empty:
        raise ValueError(f"Exame {exam_id!r} não encontrado no treino do MM-WHS.")
    record = matches.iloc[0]
    started = perf_counter()
    image = load_ccta_volume(record)
    native_reference = load_ccta_aorta_ground_truth(record)
    spacing = tuple(float(record[f"spacing_{axis}_mm"]) for axis in "xyz")
    image_data = preprocess_ccta_volume(image, spacing, config, label=native_reference)
    preprocessing_seconds = perf_counter() - started
    del image, native_reference
else:
    split_config = REPO_ROOT / "config/imagecas_splits.json"
    with split_config.open(encoding="utf-8") as stream:
        train_ids = {str(value) for value in json.load(stream)["splits"]["train"]}
    if exam_id not in train_ids:
        raise ValueError(f"Exame {exam_id!r} não encontrado no treino do ImageCAS.")

    base_path = resolve_imagecas_base_path()
    started = perf_counter()
    image_data = load_and_preprocess_image(exam_id, str(base_path), config)
    preprocessing_seconds = perf_counter() - started

lcc_image = image_data["lcc_image"]
downscale_factors = image_data["downscale_factors"]
scaled_spacing = image_data["scaled_spacing"]
reference = None
if DATASET == "mmwhs" and image_data["label"] is not None:
    reference_mask = image_data["label"] > 0
    if reference_mask.shape != lcc_image.shape:
        raise ValueError(
            "A referência MM-WHS está fora da grade da tomografia processada."
        )
    if reference_mask.any():
        reference = reference_mask

print(f"{DATASET} · treino · {exam_id} · resolução {RESOLUTION}")
print(f"Grade processada {lcc_image.shape} · espaçamento xyz (mm) {scaled_spacing}")
print(f"Carregamento e pré-processamento em {preprocessing_seconds:.2f} s")
if DATASET == "mmwhs" and reference is None:
    print("Referência 820 ausente ou vazia na grade processada.")

visual_spacing = (scaled_spacing[1], scaled_spacing[0], scaled_spacing[2])

## 3. Localização e segmentação pareadas

A Hough e o filtro seguem a mesma configuração nos dois caminhos. A variante
Frangi usa um mapa 3D calculado na tomografia pós-LCC, com `gamma` automático e
normalização entre 0 e 1. Cada trajetória filtrada inicializa o level set na
mesma tomografia pós-LCC.

In [ ]:
evaluate_variant = partial(
    evaluate_aorta_localization,
    lcc_image,
    downscale_factors,
    scaled_spacing,
    config,
    reference=reference,
    use_gpu=USE_GPU,
)
baseline_result = evaluate_variant()
frangi_result = evaluate_variant(
    frangi_config={**frangi_config, "sigmas": sigmas},
    include_vesselness=True,
)

## 4. Resultados

A cobertura axial conta as fatias com círculos após o filtro, incluindo
interpolações, em relação a todas as fatias processadas. O Dice está na escala
de 0 a 1 e a diferença é calculada como variante Frangi menos baseline.

In [ ]:
def result_row(result):
    return {
        "Status": result.status,
        "Círculos antes do filtro": result.circles_before_filter,
        "Círculos após o filtro": result.circles_after_filter,
        "Cobertura axial (%)": result.coverage_percent,
        "Volume (mL)": result.volume_ml,
        "Frangi (s)": result.frangi_seconds,
        "Hough + filtro (s)": result.localization_seconds,
        "Segmentação (s)": result.segmentation_seconds,
        "Dice vs. rótulo 820": result.dice,
    }


def show_aorta_result(result, spacing, *, title, color):
    print(title)
    if result.mask is None or not result.mask.any():
        print(result.status)
        return
    display(
        visualize_3d_k3d(result.mask, spacing=spacing, color=color, display_plot=False)
    )

In [ ]:
baseline_name = "Baseline · CT"
frangi_name = "Variante · Frangi"
summary = pd.DataFrame.from_dict(
    {
        baseline_name: result_row(baseline_result),
        frangi_name: result_row(frangi_result),
    },
    orient="index",
).rename_axis("Método")
if reference is None:
    summary = summary.drop(columns="Dice vs. rótulo 820")
else:
    dice_values = pd.to_numeric(summary["Dice vs. rótulo 820"])
    if dice_values.notna().all():
        delta = dice_values[frangi_name] - dice_values[baseline_name]
        print(f"Diferença de Dice (Frangi − baseline) {delta:+.4f}")

print(f"Sigmas da variante Frangi {sigmas}")
display(summary.round(4).fillna("—"))

### Comparação anterior de sigmas

Execução em CPU no exame de treino `ct_train_1001` do MM-WHS, em resolução `mid`. O delta dessa tabela usa a faixa de 8 a 19 como referência. A busca anterior recomendou `(8, 14, 19)` pelo tempo. A lista `(16, 17, 18, 19)` teve o maior Dice e foi a referência para a comparação abaixo.

| Sigmas (voxels) | Escalas | Status | Círculos antes do filtro | Círculos após o filtro | Cobertura axial (%) | Volume (mL) | Frangi (s) | Hough + filtro (s) | Segmentação (s) | Dice vs. rótulo 820 | Delta Dice (pp) | Total após pré-processamento (s) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| (8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19) | 12 | Máscara produzida | 131 | 131 | 36.0882 | 93.5734 | 583.1978 | 2.4301 | 12.4780 | 0.8750 | 0.0000 | 598.1058 |
| (8.0, 12.0, 16.0, 19.0) | 4 | Máscara produzida | 131 | 131 | 36.0882 | 93.6853 | 194.2210 | 2.2652 | 11.9447 | 0.8755 | 0.0503 | 208.4309 |
| (8.0, 14.0, 19.0) | 3 | Máscara produzida | 131 | 131 | 36.0882 | 93.7039 | 143.2358 | 2.4150 | 12.9393 | 0.8755 | 0.0535 | 158.5902 |
| (8.0, 9.0, 10.0, 11.0) | 4 | Máscara produzida | 131 | 131 | 36.0882 | 93.6089 | 164.1082 | 2.3292 | 12.1168 | 0.8751 | 0.0178 | 178.5542 |
| (16.0, 17.0, 18.0, 19.0) | 4 | Máscara produzida | 172 | 172 | 47.3829 | 110.7789 | 232.6204 | 3.1174 | 14.2815 | 0.9586 | 8.3679 | 250.0193 |

### Comparação dos sigmas isolados

Mesmo exame e resolução, com referência `(16, 17, 18, 19)`. O sigma `16` isolado teve o maior Dice e o menor tempo entre os testados.

| Sigmas (voxels) | Círculos após o filtro | Volume (mL) | Dice vs. rótulo 820 | Delta Dice (pp) | Total após pré-processamento (s) | Redução de tempo (%) |
| --- | --- | --- | --- | --- | --- | --- |
| (16, 17, 18, 19) | 172 | 110.7789 | 0.9586 | 0.0000 | 274.6017 | 0.0000 |
| (16,) | 174 | 110.9670 | 0.9596 | 0.0942 | 73.3773 | 73.2786 |
| (17,) | 170 | 110.5739 | 0.9576 | -0.1040 | 75.0836 | 72.6573 |
| (18,) | 168 | 110.2212 | 0.9560 | -0.2647 | 78.9816 | 71.2378 |
| (19,) | 164 | 109.2930 | 0.9519 | -0.6710 | 80.2137 | 70.7891 |

## 5. MIP axial do mapa Frangi

Esta projeção mostra o máximo ao longo das fatias do mesmo mapa de vasos
que alimenta a localização por Hough.

In [ ]:
if frangi_result.vesselness is None:
    print(frangi_result.status)
else:
    plot_mip_projection(
        frangi_result.vesselness,
        title=f"MIP axial do Frangi · {exam_id} · sigmas {sigmas} voxels",
        views=("axial",),
        cmap="grey",
        vmin=0,
        vmax=1,
        show_labels=False,
    )

## 6. Máscaras da aorta em 3D

A primeira célula mostra o baseline na tomografia. A segunda mostra a variante
Frangi com os sigmas do experimento. As duas cenas usam a grade processada em
milímetros.

In [ ]:
show_aorta_result(
    baseline_result, visual_spacing, title="Baseline · CT", color=0x39C5FF
)

In [ ]:
show_aorta_result(
    frangi_result,
    visual_spacing,
    title=f"Frangi · sigmas {sigmas} voxels",
    color=0xFF9E31,
)